![A soccer pitch for an international match.](soccer-pitch.jpg)

You're working as a sports journalist at a major online sports media company, specializing in soccer analysis and reporting. You've been watching both men's and women's international soccer matches for a number of years, and your gut instinct tells you that more goals are scored in women's international football matches than men's. This would make an interesting investigative article that your subscribers are bound to love, but you'll need to perform a valid statistical hypothesis test to be sure!

While scoping this project, you acknowledge that the sport has changed a lot over the years, and performances likely vary a lot depending on the tournament, so you decide to limit the data used in the analysis to only official `FIFA World Cup` matches (not including qualifiers) since `2002-01-01`.

You create two datasets containing the results of every official men's and women's international football match since the 19th century, which you scraped from a reliable online source. This data is stored in two CSV files: `women_results.csv` and `men_results.csv`.

The question you are trying to determine the answer to is:

> Are more goals scored in women's international soccer matches than men's?

You assume a **10% significance level**, and use the following null and alternative hypotheses:

$H_0$ : The mean number of goals scored in women's international soccer matches is the same as men's.

$H_A$ : The mean number of goals scored in women's international soccer matches is greater than men's.

In [159]:
# Start your code here!
import pandas as pd
import numpy as np 
import pingouin
import matplotlib.pyplot as plt
import seaborn as sns

In [160]:
man=pd.read_csv('men_results.csv')
man_fdf=pd.DataFrame(man)
women=pd.read_csv('women_results.csv')
women_fdf=pd.DataFrame(women)

In [161]:
man_fdf.iloc[:5,]

,Unnamed: 0,date,home_team,away_team,home_score,away_score,tournament
0,0,1872-11-30,Scotland,England,0,0,Friendly
1,1,1873-03-08,England,Scotland,4,2,Friendly
2,2,1874-03-07,Scotland,England,2,1,Friendly
3,3,1875-03-06,England,Scotland,2,2,Friendly
4,4,1876-03-04,Scotland,England,3,0,Friendly


In [162]:
women_fdf.head(5)

,Unnamed: 0,date,home_team,away_team,home_score,away_score,tournament
0,0,1969-11-01,Italy,France,1,0,Euro
1,1,1969-11-01,Denmark,England,4,3,Euro
2,2,1969-11-02,England,France,2,0,Euro
3,3,1969-11-02,Italy,Denmark,3,1,Euro
4,4,1975-08-25,Thailand,Australia,3,2,AFC Championship


In [163]:
man_fdf['tournament'].unique()

array(['Friendly', 'British Home Championship', 'Évence Coppée Trophy',
       'Copa Lipton', 'Copa Newton', 'Copa Premio Honor Argentino',
       'Copa Premio Honor Uruguayo', 'Copa Roca', 'Copa América',
       'Copa Chevallier Boutell', 'Nordic Championship',
       'Central European International Cup', 'Baltic Cup', 'Balkan Cup',
       'FIFA World Cup', 'Copa Rio Branco',
       'FIFA World Cup qualification', 'Bolivarian Games',
       'CCCF Championship', 'NAFC Championship', 'Copa Oswaldo Cruz',
       'Pan American Championship', 'Copa del Pacífico',
       "Copa Bernardo O'Higgins", 'AFC Asian Cup qualification',
       'Atlantic Cup', 'AFC Asian Cup', 'African Cup of Nations',
       'Copa Paz del Chaco', 'Merdeka Tournament',
       'UEFA Euro qualification', 'UEFA Euro',
       'Windward Islands Tournament',
       'African Cup of Nations qualification', 'Vietnam Independence Cup',
       'Copa Carlos Dittborn', 'Phillip Seaga Cup',
       'CONCACAF Championship', 'Copa Ju

In [164]:
women_fdf['tournament'].unique()

array(['Euro', 'AFC Championship', 'UEFA Euro qualification',
       'OFC Championship', 'UEFA Euro', 'African Championship',
       'CONCACAF Championship', 'Copa América', 'FIFA World Cup',
       'CONCACAF Invitational Tournament', 'Algarve Cup', 'Olympic Games',
       'Four Nations Tournament', 'African Championship qualification',
       'CONCACAF Gold Cup qualification', 'CONCACAF Gold Cup',
       'AFC Asian Cup qualification', 'AFC Asian Cup', 'Cyprus Cup',
       'Olympic Games qualification', 'Friendly', 'SheBelieves Cup',
       'FIFA World Cup qualification', 'Tournament of Nations',
       'OFC Nations Cup qualification', 'African Cup of Nations',
       'OFC Nations Cup', 'Tournoi de France',
       "Basque Country Women's Cup",
       'African Cup of Nations qualification'], dtype=object)

In [165]:
man_world_cup2002df=man_fdf[(man_fdf['tournament'
                                    ]=='FIFA World Cup')&(man_fdf['date']>='2002-01-01')]

man_world_cup2002df.iloc[:5,]

,Unnamed: 0,date,home_team,away_team,home_score,away_score,tournament
25164,25164,2002-05-31,France,Senegal,0,1,FIFA World Cup
25165,25165,2002-06-01,Germany,Saudi Arabia,8,0,FIFA World Cup
25166,25166,2002-06-01,Republic of Ireland,Cameroon,1,1,FIFA World Cup
25167,25167,2002-06-01,Uruguay,Denmark,1,2,FIFA World Cup
25168,25168,2002-06-02,Argentina,Nigeria,1,0,FIFA World Cup


In [166]:
women_world_cup20002df = women_fdf[
    (women_fdf['tournament']=='FIFA World Cup') & (women_fdf['date']>='2002-01-01')
]
print(women_world_cup20002df)
#women_fdf.head()

      Unnamed: 0        date  ... away_score      tournament
1600        1600  2003-09-20  ...          3  FIFA World Cup
1601        1601  2003-09-20  ...          0  FIFA World Cup
1602        1602  2003-09-20  ...          1  FIFA World Cup
1603        1603  2003-09-20  ...          0  FIFA World Cup
1604        1604  2003-09-21  ...          1  FIFA World Cup
...          ...         ...  ...        ...             ...
4465        4465  2019-06-29  ...          2  FIFA World Cup
4466        4466  2019-07-02  ...          2  FIFA World Cup
4467        4467  2019-07-03  ...          0  FIFA World Cup
4468        4468  2019-07-06  ...          2  FIFA World Cup
4469        4469  2019-07-07  ...          0  FIFA World Cup

[200 rows x 7 columns]


In [167]:
man_world_cup2002df['m_total_score']=man_world_cup2002df['home_score']+man_world_cup2002df['away_score']
man_world_cup2002df['m_total_score'].mean()

2.5130208333333335

In [168]:
women_world_cup20002df['W_total_score']=women_world_cup20002df['home_score'] + women_world_cup20002df['away_score']
women_world_cup20002df['W_total_score'].mean()

2.98

In [169]:
compindfm_w = man_world_cup2002df.merge(women_world_cup20002df, on='Unnamed: 0', how='outer',
                                        suffixes=('_man', '_women'))
compindfm_w.head(10)

,Unnamed: 0,date_man,home_team_man,away_team_man,home_score_man,away_score_man,tournament_man,m_total_score,date_women,home_team_women,away_team_women,home_score_women,away_score_women,tournament_women,W_total_score
0,25164,2002-05-31,France,Senegal,0.0,1.0,FIFA World Cup,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25165,2002-06-01,Germany,Saudi Arabia,8.0,0.0,FIFA World Cup,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25166,2002-06-01,Republic of Ireland,Cameroon,1.0,1.0,FIFA World Cup,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25167,2002-06-01,Uruguay,Denmark,1.0,2.0,FIFA World Cup,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25168,2002-06-02,Argentina,Nigeria,1.0,0.0,FIFA World Cup,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,25169,2002-06-02,England,Sweden,1.0,1.0,FIFA World Cup,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,25170,2002-06-02,Paraguay,South Africa,2.0,2.0,FIFA World Cup,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,25171,2002-06-02,Spain,Slovenia,3.0,1.0,FIFA World Cup,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,25172,2002-06-03,Brazil,Turkey,2.0,1.0,FIFA World Cup,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,25173,2002-06-03,Croatia,Mexico,0.0,1.0,FIFA World Cup,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [170]:
m_w=compindfm_w[['W_total_score','m_total_score']]
#m_w_pivot=m_w.pivot(colomns=)
m_w

,W_total_score,m_total_score
0,NaN,1.0
1,NaN,8.0
2,NaN,2.0
3,NaN,3.0
4,NaN,1.0
...,...,...
579,3.0,NaN
580,3.0,NaN
581,1.0,NaN
582,3.0,NaN


In [171]:
pingouin.mwu(x=m_w['W_total_score'],
               y=m_w['m_total_score'],
              alternative='greater')

,U-val,alternative,p-val,RBC,CLES
MWU,43273.0,greater,0.005107,-0.126901,0.563451


In [172]:
#result_dict={'p_val':0.0025980724,
            #'result':'reject'}
#result_dict

In [173]:
#from scipy.stats import ttest_ind

In [174]:
#t_statistic, p_value = ttest_ind(women_world_cup20002df['W_total_score'], man_world_cup2002df['m_total_score'])
#t_statistic, p_value

In [175]:
result_dict={'p_val':0.0051066098,
            'result':'reject'}
result_dict

{'p_val': 0.0051066098, 'result': 'reject'}